#### Name of author: Gema Antón
#### version: 1
- This Jupiter will be the test for creating an ETL from scratch. 
- We will analyse all types of CSV or Excel files.
- In addition, we can modify each piece of data, including nulls and duplicates.

## Proceso ETL (Extract, Transform, Load)

|  **ETL**        |  **Acción**                                 |  **Ejemplo práctico**                          |
|-----------------|----------------------------------------------|-------------------------------------------------|
| **E: Extract**  | Obtener datos de diferentes fuentes (CSV, API, SQL) | Descargar un archivo CSV con datos de ventas.   |
| **T: Transform** | Limpiar y transformar los datos              | Eliminar valores nulos, calcular promedios, convertir formatos de fecha. |
| **L: Load**     | Cargar los datos en un destino final          | Guardar los datos limpios en una base de datos o enviarlos a un dashboard. |


In [1]:
# package imports
# Data manipulation libraries
import pandas as pd
import numpy as np
# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical libraries
import scipy.stats as st
import scipy.stats as stats
from scipy.stats import shapiro, poisson, chisquare, expon, kstest
# System libraries
import os




In [ ]:

renta = r'C:/Users/ganto\Desktop/project-gema/Personal_project/Spanish_Issues/datasets/renta.csv'
ipv = r'C:/Users/ganto\Desktop/project-gema/Personal_project/Spanish_Issues/datasets/IPV.csv'

print("¿Existe el archivo?", os.path.exists(renta))
print("Ruta usada:", renta)

# Listar lo que hay en la carpeta datasets
print("\nContenido de la carpeta datasets:")
print(os.listdir(r"C:/Users/ganto\Desktop/project-gema/Personal_project/Spanish_Issues/datasets"))


¿Existe el archivo? True
Ruta usada: C:/Users/ganto\Desktop/project-gema/Personal_project/Spanish_Issues/datasets/renta.csv

Contenido de la carpeta datasets:
['50913 (2024).csv', '50913-2025.csv', 'IndiceConsumo.csv', 'IPV.csv', 'renta.csv']


In [9]:
rentadf = pd.read_csv(renta, sep=';', encoding='latin-1')
ipvdf = pd.read_csv(ipv, sep=';', encoding='latin-1')
                    
#ipv = pd.read_csv('datasets\IPV.csv', sep=';')

In [24]:


def EDA(df):
        print(f"\n----EDA process----\n ")
        display(df.info()  )
        display(df.sample(5))
        print(f"Shape of dataset:")
        print(f"{df.shape}\n")
        print(f"Columns:")
        print(f"{df.columns}\n")
        print(f"Nulls:")
        print(f"{df.isnull().sum()}\n")
        print(f"Duplicates:")
        print(f"{df.duplicated().sum()}\n")
        print(f"Main stadistics :")
        print(f"{df.describe().T}\n")
        print(f"The trend from categoric columns:\n")
        columnas_cat = df.select_dtypes(include = 'object')
        for columna in columnas_cat:
            if df[columna].isnull().any():
                print(f"Reviewing {columna}")
                print(df[columna].value_counts())  
                print("") 
        print(f"\n-----------------------------\n")


        df.info()  


        return

def transform(df1):
    new_columns = {}
    #Empy dictionary to save new column names
    for col in df1.columns:
        #For each column, we modify the name to lowercase and remove dots
        new_columns[col] = col.lower().replace(".", "")
        new_columns[col] = col.lower().replace(" ", "_")
    df1.rename(columns = new_columns, inplace = True)

    return df1 

In [11]:

def null_check(df):
    
        print("Categorical columns with null values review:")
        col_obj = df.select_dtypes(include='object').columns
        #List for saving columns with nulls

        found_obj = False  #checking if any nulls found

        for col in col_obj:
            
            por = df[col].isnull().mean() * 100
            if por > 0:
                print(f" - {col}: {por:.0f}%  of nulls")
                #The percentage of nulls is greater than 0, so we set the flag to True
                found_obj = True #Check de que ha encontrado nulos

        if not found_obj:
            print(" There are no nulls in categorical columns")


        #Ahora comprobación de columas de tipo numérico
        print(" Numerical columns with null values review:")
        col_num = df.select_dtypes(include='number').columns
        
        found_num = False
        #we set the flag to False again for numerical columns
        for col in col_num:
            por = df[col].isnull().mean() * 100

            if por > 0:
                print(f" - {col}: {por:.0f}% of nulls")
                found_num = True
        if not found_num:
            print(" There are no nulls in numerical columns")


''' 
This function will check for nulls, then, after checking the % of nulls, it decides how to handle them
In categorical variables
% Low amount---> trend
% High amount---> other category
In numerical variables
% Low amount---> trend
% High amount---> mean/median imputation

'''

' \nThis function will check for nulls, then, after checking the % of nulls, it decides how to handle them\nIn categorical variables\n% Low amount---> trend\n% High amount---> other category\nIn numerical variables\n% Low amount---> trend\n% High amount---> mean/median imputation\n\n'

In [12]:
def null_treatment(df):
    
    col_obj = df.select_dtypes(include='object').columns        
    col_num = df.select_dtypes(include='number').columns
    no_nulls = []
    delete_cols = []

    # New list to save columns, we will treat one by one the nulls, depending on the data type and % of data
    # Traversing the loop column by column
   
    for col in col_obj:
        por = df[col].isnull().mean() * 100
        if por == 0:
            no_nulls.append(col)
            # Save the column name with no nulls
        else:    
            if por < 20:
                # Low amount of nulls: impute with mode (most frequent value)
                print(f"Processing : {col} ({por:.0f}% nulls) -> Mode or Trend")
                df[col] = df[col].fillna(df[col].mode()[0])

            elif 20 <= por < 80:
                # High amount of nulls: impute with generic value
                print(f"Processing : {col} ({por:.0f}% nulls) -> New category: 'Unknown'")
                df[col] = df[col].fillna('Unknown')

            else:
                # Too many nulls: consider dropping the column
                print(f"Column {col} has  {por:.0f}% of nulls. You should consider dropping it or analyzing it separately.")
                delete_cols.append(col)
                # Save the column name to be deleted later
                

            
    for col in col_num:
        por = df[col].isnull().mean() * 100
        if por == 0:
               no_nulls.append(col)
               # Save the column name with no nulls
        
        else:    
            if por < 20:
                # Low level of nulls: impute with mean
                print(f"Processin column: {col} ({por:.0f}% of nulls) -> Mean")
                imputer = SimpleImputer(strategy='mean')
                df[col] = imputer.fit_transform(df[[col]])

            elif 21 <= por < 70:
                # Percentage of nulls to impute with KNN
                print(f"Processing : {col} ({por:.0f}% of nulls) -> KNNImputer")
                imputer_knn = KNNImputer(n_neighbors=5)
                df[col] = imputer_knn.fit_transform(df[[col]])

                # Cambio

            elif por > 71:
            # High percentage of nulls: consider dropping the column or imputing manually
                print(f"Column {col} has  {por:.0f}% of nulls. You should consider dropping it or imputing manually.")
    
    # Interacting to drop the columns with too many nulls.
    #Ask the user if they want to drop the columns with too many nulls
    if delete_cols:
        print(f"The following columns have too many nulls and are recommended to be dropped: {delete_cols}")
    
        while True:
            user_input = input("Do you want to drop these columns? (Y/N): ").strip().lower()
            
            if user_input == 'y':
                df.drop(columns=delete_cols, inplace=True)
                print("Columns dropped.")
                break
                
            elif user_input == 'n':
                print("Columns not dropped. Please consider handling them manually.")
                break
                
            else:
                print("Invalid option. Please type 'Y' or 'N'.")




                
    return df

In [18]:
EDA(rentadf)




----EDA process----
 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Comunidades y Ciudades Autónomas  100 non-null    object 
 1   Renta anual neta media por hogar  100 non-null    object 
 2   Periodo                           100 non-null    int64  
 3   Total                             100 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 3.3+ KB


None

,Comunidades y Ciudades Autónomas,Renta anual neta media por hogar,Periodo,Total
95,19 Melilla,Renta neta media por hogar,2024,39.855
25,05 Canarias,Renta neta media por hogar,2024,34.819
53,10 Comunitat Valenciana,Renta neta media por hogar,2021,27.603
36,07 Castilla y León,Renta neta media por hogar,2023,31.863
2,Total Nacional,Renta neta media por hogar,2022,32.216


Shape of dataset:
(100, 4)

Columns:
Index(['Comunidades y Ciudades Autónomas', 'Renta anual neta media por hogar',
       'Periodo', 'Total'],
      dtype='object')

Data type:
Comunidades y Ciudades Autónomas     object
Renta anual neta media por hogar     object
Periodo                               int64
Total                               float64
dtype: object

Nulls:
Comunidades y Ciudades Autónomas    0
Renta anual neta media por hogar    0
Periodo                             0
Total                               0
dtype: int64

Duplicates:
0

Main stadistics :
         count        mean       std      min         25%       50%       75%  \
Periodo  100.0  2022.00000  1.421338  2020.00  2021.00000  2022.000  2023.000   
Total    100.0    33.04101  5.180847    22.25    29.10025    32.156    36.927   

              max  
Periodo  2024.000  
Total      44.889  

The trend from categoric columns:


-----------------------------

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100

In [19]:
EDA(ipvdf)



----EDA process----
 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2960 entries, 0 to 2959
Data columns (total 6 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Total Nacional                             2960 non-null   object
 1   Comunidades y Ciudades Autónomas           2812 non-null   object
 2   General, vivienda nueva y de segunda mano  2960 non-null   object
 3   Índices y tasas                            2960 non-null   object
 4   Periodo                                    2960 non-null   object
 5   Total                                      2664 non-null   object
dtypes: object(6)
memory usage: 138.9+ KB


None

,Total Nacional,Comunidades y Ciudades Autónomas,"General, vivienda nueva y de segunda mano",Índices y tasas,Periodo,Total
2891,Nacional,19 Melilla,Vivienda segunda mano,Índice,2024T1,NaN
2647,Nacional,"17 Rioja, La",Vivienda segunda mano,Índice,2011T1,"144,599"
30,Nacional,NaN,Vivienda nueva,Índice,2017T4,"113,902"
270,Nacional,01 Andalucía,Vivienda segunda mano,Índice,2013T2,"97,458"
1202,Nacional,08 Castilla - La Mancha,Vivienda nueva,Índice,2020T4,"127,011"


Shape of dataset:
(2960, 6)

Columns:
Index(['Total Nacional', 'Comunidades y Ciudades Autónomas',
       'General, vivienda nueva y de segunda mano', 'Índices y tasas',
       'Periodo', 'Total'],
      dtype='object')

Data type:
Total Nacional                               object
Comunidades y Ciudades Autónomas             object
General, vivienda nueva y de segunda mano    object
Índices y tasas                              object
Periodo                                      object
Total                                        object
dtype: object

Nulls:
Total Nacional                                 0
Comunidades y Ciudades Autónomas             148
General, vivienda nueva y de segunda mano      0
Índices y tasas                                0
Periodo                                        0
Total                                        296
dtype: int64

Duplicates:
0

Main stadistics :
                                          count unique             top  freq
Total Nacional  

In [31]:
itvdf = transform(ipvdf)
rentadf = transform(rentadf)

In [37]:
display(itvdf.sample(5))
display(rentadf.sample(5))

,total_nacional,comunidades_y_ciudades_autónomas,"general,_vivienda_nueva_y_de_segunda_mano",índices_y_tasas,periodo,total
2560,Nacional,"17 Rioja, La",Vivienda nueva,Índice,2014T2,"98,931"
1377,Nacional,09 Cataluña,Vivienda nueva,Índice,2014T1,"91,715"
2558,Nacional,"17 Rioja, La",Vivienda nueva,Índice,2014T4,"98,219"
1758,Nacional,11 Extremadura,Vivienda segunda mano,Índice,2011T2,"131,372"
1700,Nacional,11 Extremadura,Vivienda nueva,Índice,2007T2,"123,189"


,comunidades_y_ciudades_autónomas,renta_anual_neta_media_por_hogar,periodo,total
40,08 Castilla - La Mancha,Renta neta media por hogar,2024,31.001
84,16 País Vasco,Renta neta media por hogar,2020,37.598
89,"17 Rioja, La",Renta neta media por hogar,2020,32.096
1,Total Nacional,Renta neta media por hogar,2023,34.821
37,07 Castilla y León,Renta neta media por hogar,2022,30.212


In [30]:
itvdf = null_check(itvdf)

Categorical columns with null values review:
 - comunidades_y_ciudades_autónomas: 5%  of nulls
 - total: 10%  of nulls
 Numerical columns with null values review:
 There are no nulls in numerical columns


In [38]:
#Save the new datasets without nulls
itvdf.to_csv('IPV_cleaned.csv', index=False)
rentadf.to_csv('renta_cleaned.csv', index=False)